# Appendix: Android, measured

> Notebook 104's decoder on a Pixel 10a: CPU threads, the GPU at 16 and 32 bits, where the CPU's time goes, and how
> long the GPU takes to load.

It needs the phone plugged in, and the 32-bit GPU run alone spends six minutes loading, so `nbdev-test` skips it.

In [ ]:
import pandas as pd
import torch

from text_to_speech.litert import ON_PHONE, PHONE, adb, benchmark, install, phone_decoder, to_tflite
from text_to_speech.voice import analyse

/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torchao/quantization/quant_api.py:1558: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.
W1101 22:55:41.933000 88564 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W1101 22:55:41.970000 88564 .venv/lib/python3.13/site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1101 22:55:42.021000 88564 .venv/lib/python3.13/site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclas

In [ ]:
T = 128
u = analyse("Speed comes from the voice itself, not from stretching the audio afterwards.")
source, body = phone_decoder(T)
with torch.no_grad():
    args = tuple(
        a.contiguous() for a in (u.asr[..., :T], u.F0[:, : 2 * T], u.N[:, : 2 * T], u.style, source(u.F0[:, : 2 * T]))
    )
print(f"largest error against PyTorch: {to_tflite(body, args, f'body{T}'):.1e}")
print(adb("shell", "getprop", "ro.product.model").strip(), "with", adb("shell", "getprop", "ro.soc.model").strip())
install(PHONE / "benchmark_model")
install(PHONE / f"body{T}.tflite")

largest error against PyTorch: 6.5e-06
Pixel 10a with Tensor G4


/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/export/pt2_archive/_package.py:415: UserWarning: No complete tensor found in the group! Returning the first one. This may cause issues when your weights are not on CPU.
  complete_tensor = get_complete_tensor(group, weights_dict)


## Speed

Each call decodes 3.2 s of audio. The GPU runs without its program cache here, so its load time is the compile.

In [ ]:
adb("shell", "rm", "-rf", f"{ON_PHONE}/gpu_cache")
rows = []
for name, flags in [
    ("CPU, 1 thread", ["--num_threads=1"]),
    ("CPU, 4 threads", ["--num_threads=4"]),
    ("CPU, 8 threads", ["--num_threads=8"]),
    ("GPU, 16-bit", ["--use_gpu=true"]),
    ("GPU, 32-bit", ["--use_gpu=true", "--gpu_precision_loss_allowed=false"]),
]:
    load, per_call, _ = benchmark(f"body{T}.tflite", *flags, runs=5 if "32" in name else 10)
    rows.append({"where": name, "load (s)": load, "per call (s)": per_call, "x real time": T * 0.025 / per_call})
pd.DataFrame(rows).round(2)

,where,load (s),per call (s),x real time
0,"CPU, 1 thread",0.33,3.64,0.88
1,"CPU, 4 threads",0.40,2.27,1.41
2,"CPU, 8 threads",0.41,2.51,1.28
3,"GPU, 16-bit",107.29,0.30,10.64
4,"GPU, 32-bit",335.63,0.80,3.99


The CPU only just keeps up, with four threads beating eight: the Pixel's four smallest cores slow the others down.
The GPU at 16 bits is about seven times faster.

## Where the CPU's time goes

In [ ]:
_, _, out = benchmark(f"body{T}.tflite", "--num_threads=4", "--enable_op_profiling=true", runs=2)
print("\n".join(line.strip() for line in out.split("Summary by node type")[-1].splitlines()[1:8]))

[Node type]	  [count]	  [avg ms]	    [avg %]	    [cdf %]	  [mem KB]	[times called]
Convolution (NHWC, F32) IGEMM	       18	  1685.387	    60.407%	    60.407%	     0.000	       60
SIN	       50	   623.665	    22.353%	    82.760%	     0.000	       50
Binary Elementwise (ND)	      178	   266.222	     9.542%	    92.301%	     0.000	      628
Transpose (ND, X32) Transpose	       46	   106.186	     3.806%	    96.107%	     0.000	      134
Deconvolution (NHWC, F32) Subconv2D	        4	    42.129	     1.510%	    97.617%	     0.000	        4
Mean (ND) Mean	       37	    31.461	     1.128%	    98.745%	     0.000	      108


Convolutions, and a fifth of it in `SIN`: the decoder's "Snake" activation, `x + sin(a·x)² / a`, runs on every sample
of audio, and the phone's CPU is slow at sines. For a model of our own: a cheaper activation.

## Loading

The GPU compiles programs for the model's shapes on first load. LiteRT can save them; an app would do this once.

In [ ]:
cache = ["--use_gpu=true", f"--delegate_serialize_dir={ON_PHONE}/gpu_cache", f"--delegate_serialize_token=body{T}"]
adb("shell", "mkdir", "-p", f"{ON_PHONE}/gpu_cache")
for attempt in ["first", "second"]:
    load, per_call, _ = benchmark(f"body{T}.tflite", *cache, runs=5)
    print(f"{attempt} load: {load:.1f}s, then {per_call:.2f}s per call")

first load: 187.6s, then 0.30s per call
second load: 0.8s, then 0.30s per call
